# Spark ML: воспроизводимая базовая модель


Показать разделение данных, конвейер подготовки, метрики, простую базовую линию и коэффициенты.

Нужны `SparkSessionContext`, `pyspark.ml` и таблица `lakehouse.raw.pg__matches`; без Spark или таблицы модель не обучается. Все вычисления выполняются внутри собственной ячейки.

Ограничения: Для небольшого воспроизводимого эксперимента до отдельного процесса обучения. Фиксируйте снимок, разделение, признаки и метрики; не публикуйте модель из справочного ноутбука.


## Основные понятия
Модель строится только для демонстрации API. Признаки финального счёта создают утечку для предматчевого прогноза.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

import pandas as pd

from lakehouse_spark import SparkSessionContext

## Примеры


**1. Разделение**

Подготовьте ограниченный набор и разделите его с фиксированным зерном.



In [ ]:
with SparkSessionContext('example-ml-split') as spark:
    data = (
        spark
            .table("lakehouse.raw.pg__matches")
            .select("match_id", "radiant_win", "duration", "radiant_score", "dire_score")
            .dropna()
            .withColumn("label", F.col("radiant_win").cast("double"))
    )
    train, test = data.randomSplit([0.8, 0.2], seed=42)
    
    print("Обучение:", train.count(), "проверка:", test.count())


**2. Конвейер**

Соберите преобразование признаков и логистическую регрессию в один конвейер. Вывод покажет два обученных этапа: подготовку вектора и классификатор.



In [ ]:
with SparkSessionContext("example-ml-pipeline") as spark:
    data = (
        spark
            .table("lakehouse.raw.pg__matches").select("radiant_win", "duration", "radiant_score", "dire_score")
            .dropna()
            .withColumn("label", F.col("radiant_win").cast("double"))
            .withColumn("score_diff", F.col("radiant_score") - F.col("dire_score"))
    )
    assembler = VectorAssembler(inputCols=["duration", "score_diff"], outputCol="features")
    model = Pipeline(stages=[assembler, LogisticRegression(maxIter=20, regParam=0.1)]).fit(data)
    
    print([stage.__class__.__name__ for stage in model.stages])


**3. Метрики**

Обучите модель на одной части данных, получите прогнозы на другой и вычислите площадь под ROC. Значение около 0,5 соответствует случайному ранжированию, а значение ближе к 1 показывает лучшее разделение классов.



In [ ]:
with SparkSessionContext("example-ml-metric") as spark:
    data = (
        spark
            .table("lakehouse.raw.pg__matches")
            .select("radiant_win", "duration", "radiant_score", "dire_score")
            .dropna()
            .withColumn("label", F.col("radiant_win").cast("double"))
            .withColumn("score_diff", F.col("radiant_score") - F.col("dire_score"))
    )
    assembled = VectorAssembler(inputCols=["duration", "score_diff"], outputCol="features").transform(data)
    train, test = assembled.randomSplit([0.8, 0.2], seed=42)
    model = LogisticRegression(maxIter=20, regParam=0.1).fit(train)
    predictions = model.transform(test)
    metric = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)
    
    print("ROC AUC:", round(metric, 4))


**4. Базовая линия**

Сравните модель с постоянным предсказанием наиболее частого класса.



In [ ]:
labels = pd.Series([1, 1, 0, 1, 0])
majority = labels.mode().iloc[0]
print("Наиболее частый класс:", majority, "доля:", (labels == majority).mean())

**5. Коэффициенты**

Обучите линейный классификатор и сопоставьте коэффициенты с порядком признаков. Знак показывает направление связи с вероятностью класса, но не доказывает причинность.



In [ ]:
with SparkSessionContext("example-ml-coefficients") as spark:
    names = ["duration", "score_diff"]
    data = (
        spark
            .table("lakehouse.raw.pg__matches")
            .select("radiant_win", "duration", "radiant_score", "dire_score").dropna()
            .withColumn("label", F.col("radiant_win").cast("double"))
            .withColumn("score_diff", F.col("radiant_score") - F.col("dire_score"))
    )
    assembled = VectorAssembler(inputCols=names, outputCol="features").transform(data)
    model = LogisticRegression(maxIter=20, regParam=0.1).fit(assembled)
    
    print(dict(zip(names, model.coefficients.toArray())))
    print("Свободный член:", model.intercept)
